# Semantic cache

Builds an embedding-based cache (FAISS + `sentence-transformers/all-MiniLM-L6-v2`) so a new prompt can reuse an earlier response if it's close enough in meaning, and measures hit rate / precision-recall / latency on the cleaned test set across several similarity thresholds. Use the Python environment from `requirements.txt`.

In [ ]:
from pathlib import Path
import importlib.util
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts/evaluate_semantic_cache.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'scripts/evaluate_semantic_cache.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

def load(module_name, filename):
    spec = importlib.util.spec_from_file_location(module_name, ROOT / f'scripts/{filename}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

semantic_cache = load('semantic_cache', 'semantic_cache.py')
recover_test_responses = load('recover_test_responses', 'recover_test_responses.py')
evaluate_semantic_cache = load('evaluate_semantic_cache', 'evaluate_semantic_cache.py')
SemanticCache = semantic_cache.SemanticCache

## Why an embedding model, not TF-IDF

TF-IDF (already used elsewhere in this repo) measures literal word overlap, so it misses paraphrases that share few words and can even rate unrelated sentences as more similar than real paraphrases. A quick, real check:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pairs = [
    ('How do I bake a chocolate cake?', 'What is the recipe for making a chocolate cake?'),
    ('How do I bake a chocolate cake?', 'How do I fix my broken laptop?'),
]
all_texts = [t for pair in pairs for t in pair]
tfidf_vecs = TfidfVectorizer().fit_transform(all_texts)
embedder = SemanticCache(threshold=0.85)

for i, (a, b) in enumerate(pairs):
    t_sim = cosine_similarity(tfidf_vecs[2*i], tfidf_vecs[2*i+1])[0][0]
    cache = SemanticCache(threshold=0.85, model=embedder.model)  # reuse the loaded model, fresh empty cache
    cache.store(a, 'n/a')
    e_sim = cache.lookup(b).similarity
    print(f'{a!r} vs {b!r}')
    print(f'  TF-IDF cosine similarity:    {t_sim:.3f}')
    print(f'  Embedding cosine similarity: {e_sim:.3f}\n')

## Recover response text

`data/test_clean.csv` only has `user_input`/`label`; the cache needs real response text, so this rejoins the raw zip's `model_output` back in. See `docs/semantic-cache.md` for the documented join caveat (24 prompts with ambiguous historical responses).

In [ ]:
responses_dir = ROOT / 'data/processed/semantic_cache'
recovered = recover_test_responses.recover(ROOT / 'data/test_clean.csv', ROOT / 'data/lmsys_toxicchat.zip')
responses_dir.mkdir(parents=True, exist_ok=True)
responses_path = responses_dir / 'test_with_responses.csv'
recovered.to_csv(responses_path, index=False)
print(f'Recovered responses for {len(recovered)} rows -> {responses_path}')

## Evaluate: hit rate, precision/recall against duplicate ground truth, latency

Two measurements across a threshold sweep, since the test set has few exact repeats — see `docs/semantic-cache.md` for why both are needed. This cell takes a minute or two (one batched embedding pass over 4,883 prompts, reused across thresholds).

In [ ]:
report = evaluate_semantic_cache.evaluate(responses_path, responses_dir)

print(f"Model: {report['model_name']}  |  test rows: {report['test_rows']}")
print(f"Normalized duplicate groups with >1 row: {report['normalized_duplicate_groups_with_more_than_one_row']}\n")
print(f"{'threshold':>9}  {'hit_rate':>9}  {'precision':>9}  {'recall':>9}")
for t in evaluate_semantic_cache.THRESHOLDS:
    s = report['streaming_simulation'][str(t)]
    d = report['duplicate_ground_truth_validation'][str(t)]
    p = f"{d['precision']:.3f}" if d['precision'] is not None else 'n/a'
    r = f"{d['recall']:.3f}" if d['recall'] is not None else 'n/a'
    print(f"{t:>9.2f}  {s['hit_rate']:>9.3f}  {p:>9}  {r:>9}")

In [ ]:
lat = report['latency_at_report_threshold']
print(f"Lookup latency at threshold {report['report_threshold']} (real, end-to-end incl. embedding):")
print(f"  mean={lat['mean_seconds']*1000:.2f}ms  median={lat['median_seconds']*1000:.2f}ms  "
      f"p95={lat['p95_seconds']*1000:.2f}ms  max={lat['max_seconds']*1000:.2f}ms")

## Sample hit review

A few of the saved hit sample, for a quick manual sanity check that the reused response actually fits the new prompt (the full 20-item sample is in the saved JSON).

In [ ]:
for item in report['hit_sample_at_report_threshold'][:5]:
    print('NEW    :', item['new_prompt'][:100])
    print('MATCHED:', item['matched_prompt'][:100])
    print('SIM    :', round(item['similarity'], 3))
    print('RESPONSE:', item['reused_response'][:100])
    print()

See [the semantic cache guide](../docs/semantic-cache.md) for the full results table, why recall being 1.000 at every threshold is a weaker result than it looks (the ground-truth duplicates are near-identical text, not true reworded paraphrases), why no threshold is declared "the" answer here, and the rest of the limitations.